In [1]:
#========================================================================
# Name: calc_wrf_gamma_properties.ipynb
# Author: McKenna W. Stanford
# Author Contact: mckenna.stanford@pnnl.gov
# Date Created: 04/17/2025
#
# Utility: Reads in WRF 3D files, limits to domain covered by CSAPR during
# CACTI, and computes necessary gamma distribution
# parameters (generally, slope and intercept parameters), along with some 
# characteristics of the PSD (e.g., MVD, MMD, spectral width, skewness)
# , and saves them along with needed microphysical variables to a new file.
#========================================================================

In [1]:
# Imports
#===============================
import xarray as xr
import numpy as np
import glob
import datetime
import time
import os
import dask
from dask.distributed import wait
from distributed import Client, LocalCluster
from functions import find_nearest, to_datetime
import pickle
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.spatial import cKDTree
import scipy
from scipy.special import gamma
from scipy.special import gammaincinv
from scipy.special import gammaincc
import dask.array as da
import numba
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [14]:
sim_date = '20181129'
domain = 'd2'
ens = 'gefs03'

path = f'/pscratch/sd/m/mckenna/wrf_lasso/coarse_grained/{sim_date}/{ens}/{domain}/cld_met_files/'

cld_files = sorted(glob.glob(path+'*_cld_*.nc'))#[::3]
met_files = sorted(glob.glob(path+'*_met_*.nc'))#[::3]
num_cld_files = len(cld_files)
num_met_files = len(met_files)
print('# of cld files:',num_cld_files)
print('# of met files:',num_met_files)

cld_datetimes = []
for ii in range(num_cld_files):
    #print(cld_files[ii])
    if domain != 'd2':
        dum_str = cld_files[ii].split('/')[-1].split('_')[-1].split('.')
        date_str = dum_str[2]
        time_str = dum_str[3]
        cld_datetimes.append(datetime.datetime(int(date_str[0:4]),int(date_str[4:6]),int(date_str[6:8]),int(time_str[0:2]),int(time_str[2:4]),0,tzinfo=datetime.timezone.utc))
    else:
        dum_str = cld_files[ii].split('/')[-1].split('.')
        date_str = dum_str[-3]
        time_str = dum_str[-2]
        cld_datetimes.append(datetime.datetime(int(date_str[0:4]),int(date_str[4:6]),int(date_str[6:8]),int(time_str[0:2]),int(time_str[2:4]),0,tzinfo=datetime.timezone.utc))

# of cld files: 49
# of met files: 49


# Helper Functions

## Subset 2D variables within radar domain

In [2]:
def subset_var(data,mask):

    # mask values where the csapr mask == 0.
    masked_data = np.ma.masked_array(data, mask=mask == 0)

    # Extract the 2D subset defined by the mask
    # The unmasked elements will define the dimensions
    unmasked_data = masked_data[~masked_data.mask]  # Extract non-masked data


    row_indices, col_indices = np.where(~masked_data.mask)  # Get the unmasked indices

    new_rows = len(np.unique(row_indices))  # Number of unique rows in the unmasked region
    new_cols = len(np.unique(col_indices))  # Number of unique columns in the unmasked region

    reshaped_data = unmasked_data.reshape((new_rows, new_cols))

    return reshaped_data.data

## Subset 3D variables within radar domain

In [3]:
def subset_var_3d(data,mask):

    nz = len(data[:,0,0])
    # mask values where the csapr mask == 0.
    reshaped_data_3d = []
    
    for kk in range(nz):
        data_2d = data[kk,:,:]
        masked_data = np.ma.masked_array(data_2d, mask=mask == 0)
        #print(aaaaa)
        
        # Extract the 2D subset defined by the mask
        # The unmasked elements will define the dimensions
        unmasked_data = masked_data[~masked_data.mask]  # Extract non-masked data
        
        
        row_indices, col_indices = np.where(~masked_data.mask)  # Get the unmasked indices
        
        new_rows = len(np.unique(row_indices))  # Number of unique rows in the unmasked region
        new_cols = len(np.unique(col_indices))  # Number of unique columns in the unmasked region
        
        reshaped_data = unmasked_data.reshape((new_rows, new_cols))
    
        reshaped_data_3d.append(reshaped_data.data)

    reshaped_data_3d = np.asarray(reshaped_data_3d)
    
    return reshaped_data_3d

## Limit domain to encompass the CSAPR Radar

In [4]:
csapr_path = '/pscratch/sd/m/mckenna/cacti/csapr_regridded/'
csapr_files = sorted(glob.glob(csapr_path+'/*.nc'))
#ds = xr.open_dataset(csapr_files[0])

In [5]:
def get_csapr_mask(csapr_file, cld_file):
    """
    Creates a rectangular mask for the CLD grid based on the CSAPR domain.
    The final mask is a perfect rectangle in grid-index space.
    """
    with xr.open_dataset(csapr_file) as ds_csapr:
        lon_csapr = ds_csapr['point_longitude'].squeeze()
        lat_csapr = ds_csapr['point_latitude'].squeeze()

    with xr.open_dataset(cld_file) as ds_cld:
        lon_cld = ds_cld['XLONG'].squeeze()
        lat_cld = ds_cld['XLAT'].squeeze()

    # --- Step 1: Create the initial geographic mask (as before) ---
    lat_min_csapr = lat_csapr.min().item()
    lat_max_csapr = lat_csapr.max().item()
    lon_min_csapr = lon_csapr.min().item()
    lon_max_csapr = lon_csapr.max().item()

    geographic_mask = (
        (lon_cld >= lon_min_csapr) & (lon_cld <= lon_max_csapr) &
        (lat_cld >= lat_min_csapr) & (lat_cld <= lat_max_csapr)
    ).values  # Work with numpy array from here

    # --- Step 2: Find the min/max row and column indices of the mask ---
    # .any(axis=0) -> checks which COLUMNS have at least one 'True'
    # .any(axis=1) -> checks which ROWS have at least one 'True'
    
    # Find the first and last column index that contain a valid point
    cols_with_data = np.where(geographic_mask.any(axis=0))[0]
    if len(cols_with_data) == 0: # Handle case where there's no overlap
        return np.zeros_like(lon_cld.values, dtype=float)
    j_min, j_max = cols_with_data.min(), cols_with_data.max()

    # Find the first and last row index that contain a valid point
    rows_with_data = np.where(geographic_mask.any(axis=1))[0]
    i_min, i_max = rows_with_data.min(), rows_with_data.max()

    # --- Step 3 & 4: Create a new mask and fill the rectangular region ---
    rectangular_mask = np.zeros_like(geographic_mask, dtype=float)
    # Use slicing to fill the bounding box in index space
    rectangular_mask[i_min : i_max + 1, j_min : j_max + 1] = 1.0
    
    # Return the final rectangular mask and the corner indices for visualization
    corner_indices = {'i_min': i_min, 'i_max': i_max, 'j_min': j_min, 'j_max': j_max}
    #return rectangular_mask, lon_cld.values, lat_cld.values, corner_indices
    return rectangular_mask

In [6]:
def get_3d_vars(cld_file,met_file,csapr_file):
    #=======================
    # Load 3D variables
    #=======================
    ds_cld = xr.open_dataset(cld_file)
    ds_met = xr.open_dataset(met_file)

    temp = ds_met['TEMPERATURE'].values.squeeze()-273.15 # deg C
    pressure = ds_met['PRESSURE'].values.squeeze()*1.e2 # Pa
    qv = ds_met['QVAPOR'].values.squeeze() # kg/kg
    w = ds_met['WA'].values.squeeze() # m/s
    z = ds_met['HAMSL'].values.squeeze() # meters
    lon = ds_met['XLONG'].values.squeeze()
    lat = ds_met['XLAT'].values.squeeze()
    np_time = ds_met['Time'].values[0]
    pd_time = pd.to_datetime(np_time)
    time_dt = pd_time.to_pydatetime().replace(tzinfo=datetime.timezone.utc)

    
    qc = ds_cld['QCLOUD'].values.squeeze() # kg/kg
    qi = ds_cld['QICE'].values.squeeze() # kg/kg
    qs = ds_cld['QSNOW'].values.squeeze() # kg/kg
    qr = ds_cld['QRAIN'].values.squeeze() # kg/kg 
    qg = ds_cld['QGRAUP'].values.squeeze() # kg/kg
    nc = ds_cld['QNCLOUD'].values.squeeze() # /kg
    ni = ds_cld['QNICE'].values.squeeze() # /kg
    nr = ds_cld['QNRAIN'].values.squeeze() # /kg
    
    ds_cld.close()
    ds_met.close()
    
    csapr_mask = get_csapr_mask(csapr_file,cld_file)
    temp = subset_var_3d(temp,csapr_mask)
    pressure = subset_var_3d(pressure,csapr_mask)
    qv = subset_var_3d(qv,csapr_mask)
    w = subset_var_3d(w,csapr_mask)
    z = subset_var_3d(z,csapr_mask)
    qc = subset_var_3d(qc,csapr_mask)
    qi = subset_var_3d(qi,csapr_mask)
    qs = subset_var_3d(qs,csapr_mask)
    qr = subset_var_3d(qr,csapr_mask)
    qg = subset_var_3d(qg,csapr_mask)
    nc = subset_var_3d(nc,csapr_mask)
    ni = subset_var_3d(ni,csapr_mask)
    nr = subset_var_3d(nr,csapr_mask)
    lon = subset_var(lon,csapr_mask)
    lat = subset_var(lat,csapr_mask)

    
    #=======================
    # Air density
    #=======================
    Rd=287.04
    Tv = (temp+237.15)*(1. + 0.61*qv)
    rho_air = pressure/(Rd*Tv)


    out_dict = {'qc':qc,\
                'qr':qr,\
                'qi':qi,\
                'qs':qs,\
                'qg':qg,\
                'nc':nc,\
                'nr':nr,\
                'ni':ni,\
                'rho_air':rho_air,\
                'zamsl':z,\
                'temp':temp,\
                'pressure':pressure,\
                'w':w,\
                'lon':lon,\
                'lat':lat,\
                'time_dt':time_dt,\
               }

    return out_dict

In [7]:
def compute_gamma_dsd_parameters_liquid_3d(q, N, rho_air, species, mu=0.0, rho_w=1000.0, q_min_threshold=1e-12, N_min_threshold=1e-6):

    from scipy.special import gamma
    from scipy.special import gammaincinv
    from scipy.special import gammaincc
    """
    Computes DSD diagnostic parameters for a 3D array of liquid species (rain, cloud, etc.) 
    including median mass diameter, reflectivity, and both intercept and slope parameters.
    
    Parameters:
    - q: 3D array of liquid species water mixing ratio [kg/kg]
    - N: 3D array of liquid species number concentration [#/kg]
    - rho_air: 3D array of moist air density [kg/m^3]
    - mu: shape parameter of the gamma distribution (default = 0 for exponential)
    - rho_w: density of water [kg/m^3] (default = 1000)
    - q_min_threshold: Minimum threshold for q to perform the calculation (default = 1e-12 kg/m^3, per Thompson scheme)
    - N_min_threshold: Minimum threshold for N to perform the calculation (default = 1e-6 /m^3, per Thompson scheme)
    
    Returns:
    - Dictionary containing 3D arrays for N_0 (intercept), lambda (slope), D_num, D_vol, D_mass, 
      spectral widths, skewnesses, median mass diameter, and reflectivity (Z)
    """

    # Apply threshold condition directly to q_liquid and N_liquid arrays
    valid_mask = (q*rho_air > q_min_threshold) & (N*rho_air > N_min_threshold)

    # Initialize arrays with NaN values
    N_0 = np.full_like(q, np.nan, dtype=np.float64)
    log_N_0 = np.full_like(q, np.nan, dtype=np.float64)
    lambda_ = np.full_like(q, np.nan, dtype=np.float64)
    D_num = np.full_like(q, np.nan, dtype=np.float64)
    D_vol = np.full_like(q, np.nan, dtype=np.float64)
    D_mass = np.full_like(q, np.nan, dtype=np.float64)
    #sigma_num = np.full_like(q, np.nan, dtype=np.float64)
    #skew_num = np.full_like(q, np.nan, dtype=np.float64)
    #kurt_num = np.full_like(q, np.nan, dtype=np.float64)
    #sigma_mass = np.full_like(q, np.nan, dtype=np.float64)
    #skew_mass = np.full_like(q, np.nan, dtype=np.float64)
    #kurt_mass = np.full_like(q, np.nan, dtype=np.float64)
    median_mass_diameter = np.full_like(q, np.nan, dtype=np.float64)
    Z = np.full_like(q, np.nan, dtype=np.float64)
    dBZ = np.full_like(q, np.nan, dtype=np.float64)
    q_gt_100um = np.full_like(q,np.nan,dtype=np.float64)
    N_gt_100um = np.full_like(q,np.nan,dtype=np.float64)


    # Save original variables
    q_orig = q.copy()
    N_orig = N.copy()
    rho_air_orig = rho_air.copy()

    
    # For the valid indices (where q and N are above threshold)
    q = q[valid_mask]
    N = N[valid_mask]
    rho_air = rho_air[valid_mask]

    # If cloud, need to compute mu; it will be a 3D array; if rain, mu is an integer
    if species == 'cloud':
        dum = 1000.e6 / (N*rho_air) + 2
        dum_int = np.round(dum).astype(int)
        mu = np.minimum(15, dum_int)
        

    # Compute lambda and N_0 based on q_liquid and N_liquid
    am = (np.pi * rho_w) / 6.0 # pre-factor in m-D relationship for liquid [kg]
    bm = 3. # exponent in m-D relatinship for liquid

    easy_option = False
    if easy_option:
        lambda_ij = (gamma(mu + bm + 1) / gamma(mu + 1) * (N / q) * am) ** (1 / bm) # [m^-1]
        N_0_ij = (N*rho_air)*lambda_ij ** (mu + 1) / (gamma(mu + 1)) # [m^-4]
    else:
        if species == 'cloud':
            Nt_c_max = 1999.E6
            D0c = 1.e-6
            D0r = 50.e-6
            rc = q*rho_air
            tmp_N = np.maximum(2.,np.minimum(N*rho_air,Nt_c_max))
            lamc = (tmp_N*gamma(mu + bm + 1)*am/rc/gamma(mu + 1))**(1./bm)
            xDc = (bm + mu + 1.)/lamc
            dumid1 = np.where(xDc < D0c)
            dumid2 = np.where(xDc > D0r*2.)
            if np.size(dumid1) > 0.:
                dumid1 = dumid1[0]
                lamc[dumid1] = (bm + mu[dumid1] + 1.)/D0c
            elif np.size(dumid2) > 0.:
                dumid2 = dumid2[0]
                lamc[dumid2] = (bm + mu[dumid2] + 1.)/(D0r*2.)
            tmp_N = np.minimum(Nt_c_max,gamma(mu + 1)/gamma(mu + bm + 1)*rc/am*lamc**bm)
            
            lambda_ij = lamc
            N_0_ij = tmp_N*lambda_ij**(mu+1)/gamma(mu+1.)
        
        else:
            # Rain stays easy
            lambda_ij = (gamma(mu + bm + 1) / gamma(mu + 1) * (N / q) * am) ** (1 / bm) # [m^-1]
            N_0_ij = (N*rho_air)*lambda_ij ** (mu + 1) / (gamma(mu + 1)) # [m^-4]
        
    log_N_0_ij = np.log(N_0_ij)


    # Compute moments M0 to M7
    M = {}
    for n in range(8):
        M[n] = N_0_ij * gamma(mu + n + 1) / lambda_ij ** (mu + n + 1)

    # Number-weighted statistics (D_num, D_vol, sigma_num, skew_num, kurt_num)
    D_num_ij = M[1] / M[0]
    D_vol_ij = (M[3] / M[0]) ** (1/3)
    
    # Central moments (number-weighted)
    mu2_num = M[2] / M[0] - D_num_ij**2
    mu3_num = M[3] / M[0] - 3 * D_num_ij * mu2_num - D_num_ij**3
    mu4_num = M[4] / M[0] - 4 * D_num_ij * mu3_num - 6 * D_num_ij**2 * mu2_num - D_num_ij**4
    
    #sigma_num_ij = np.sqrt(mu2_num)
    #skew_num_ij = mu3_num / sigma_num_ij**3
    #kurt_num_ij = mu4_num / sigma_num_ij**4 - 3.0  # excess kurtosis
    
    # Mass-weighted statistics (D_mass, sigma_mass, skew_mass, kurt_mass)
    D_mass_ij = M[4] / M[3]
    
    # Central moments (mass-weighted)
    mu2_mass = M[5] / M[3] - D_mass_ij**2
    mu3_mass = M[6] / M[3] - 3 * D_mass_ij * mu2_mass - D_mass_ij**3
    mu4_mass = M[7] / M[3] - 4 * D_mass_ij * mu3_mass - 6 * D_mass_ij**2 * mu2_mass - D_mass_ij**4
    
    #sigma_mass_ij = np.sqrt(mu2_mass)
    #skew_mass_ij = mu3_mass / sigma_mass_ij**3
    #kurt_mass_ij = mu4_mass / sigma_mass_ij**4 - 3.0  # excess kurtosis

    # Median mass diameter using incomplete gamma function
    median_mass_diameter_ij = gammaincinv(mu + bm + 1, 0.5) / lambda_ij # m

    # Compute the 6th moment for reflectivity (Z)
    M_6 = N_0_ij * gamma(mu + 6 + 1) / lambda_ij ** (mu + 6 + 1) # m^6/m^3
    K = 1.e18
    Z_ij = K*M_6  # mm^6/m^3
    dBZ_ij = 10.*np.log10(Z_ij)

    # Mass & number concentration above 100 um
    D_thresh = 100.*1.e-6 # 100 um in meters
    s_num = mu + 1.
    s_mass = mu + bm + 1.
    x = lambda_ij * D_thresh
    
    # Number concentration above 100 microns [#/m^3]
    N_gt_100um_ij = N_0_ij / lambda_ij**s_num * gamma(s_num) * gammaincc(s_num, x)
    
    # Mass concentration above 100 microns [kg/m^3]
    q_gt_100um_ij = N_0_ij * am / lambda_ij**s_mass * gamma(s_mass) * gammaincc(s_mass, x)

    # Store values in the valid array positions
    N_0[valid_mask] = N_0_ij
    log_N_0[valid_mask] = log_N_0_ij
    lambda_[valid_mask] = lambda_ij
    D_num[valid_mask] = D_num_ij
    D_vol[valid_mask] = D_vol_ij
    D_mass[valid_mask] = D_mass_ij
    #sigma_num[valid_mask] = sigma_num_ij
    #skew_num[valid_mask] = skew_num_ij
    #kurt_num[valid_mask] = kurt_num_ij
    #sigma_mass[valid_mask] = sigma_mass_ij
    #skew_mass[valid_mask] = skew_mass_ij
    #kurt_mass[valid_mask] = kurt_mass_ij
    median_mass_diameter[valid_mask] = median_mass_diameter_ij
    Z[valid_mask] = Z_ij
    dBZ[valid_mask] = dBZ_ij
    q_gt_100um[valid_mask] = q_gt_100um_ij/rho_air
    N_gt_100um[valid_mask] = N_gt_100um_ij/rho_air
    
    #q_orig = np.full_like(N_0, np.nan, dtype=np.float64)
    #N_orig = np.full_like(N_0, np.nan, dtype=np.float64)
    #rho_air_orig = np.full_like(N_0, np.nan, dtype=np.float64)
    #q_orig[valid_mask] = q
    #N_orig[valid_mask] = N
    #rho_air_orig[valid_mask] = rho_air

    M_full = {}
    for key in M:
        M_full[key] = np.full_like(N_0,np.nan,dtype=np.float64)
        M_full[key][valid_mask] = M[key]
    
    if species == 'cloud':
        tmp_mu = np.full_like(N_0,np.nan)
        tmp_mu[valid_mask] = mu
        mu = tmp_mu

    return {
        'N_0': N_0,  # intercept parameter [m^-4]
        'log_N_0': log_N_0,  # Log of intercept parameter [log(m^-4)]
        'lambda': lambda_,  # Slope parameter [m^-1]
        'D_num': D_num,  # Number-weighted mean diameter [m]
        'D_vol': D_vol,  # Volume-weighted mean diameter [m]
        'D_mass': D_mass,  # Mass-weighted mean diameter [m]
        #'sigma_num': sigma_num,  # Number-weighted standard deviation [m]
        #'skew_num': skew_num,  # Number-weighted skewness [-]
        #'kurt_num': kurt_num,  # Number-weighted kurtosis [-]
        #'sigma_mass': sigma_mass,  # Mass-weighted standard deviation [-]
        #'skew_mass': skew_mass,  # Mass-weighted skewness [-]
        #'kurt_mass': kurt_mass,  # Mass-weighted kurtosis [-]
        'mmd': median_mass_diameter,  # Median mass diameter [m]
        'Z': Z,  # Linear Reflectivity [mm^6 m^-3]
        'dBZ': dBZ,  # Reflectivity [dBZ]
        'mu':mu, # Shape parameter (3D for cloud, integer for rain) [-]
        'q':q_orig, # prongostic mass mixing ratio [kg/kg]
        'N':N_orig, # prognsotic total number concentraiton mixing ratio [#/kg]
        'rho_air':rho_air_orig, #  moist air density [kg/m^3]
        'q_gt_100um':q_gt_100um, # mass mixing ratio for diameters > 100 um [kg/kg] 
        'N_gt_100um':N_gt_100um, # number concentration mixing ratio for diameters > 100 um [#/kg] 
        'M_dict':M_full, # dictionary of raw moments
    }

In [21]:
def write_file(x3d_dict,rain_dict,cloud_dict,liq_dict):
    out_dict = {'D_num':liq_dict['D_num'],
            'D_vol':liq_dict['D_vol'],
            'D_mass':liq_dict['D_mass'],
            #'sigma_num':liq_dict['sigma_num'],
            #'skew_num':liq_dict['skew_num'],
            #'kurt_num':liq_dict['kurt_num'],
            #'sigma_mass':liq_dict['sigma_mass'],
            #'skew_mass':liq_dict['skew_mass'],
            #'kurt_mass':liq_dict['kurt_mass'],
            'mmd':liq_dict['mmd'],
            'N_gt_100um':liq_dict['N_gt_100um'],
            'q_gt_100um':liq_dict['q_gt_100um'],
            'q':liq_dict['q'],
            'N':liq_dict['N'],
            'rho_air':liq_dict['rho_air'],
            'temp':x3d_dict['temp'],
            'N_0_r':rain_dict['N_0'],
            'N_0_c':cloud_dict['N_0'],
            'lambda_r':rain_dict['lambda'],
            'lambda_c':cloud_dict['lambda'],
            'mu_c':cloud_dict['mu'],
            'D_num_c':cloud_dict['D_num'],
            'D_vol_c':cloud_dict['D_vol'],
            'D_mass_c':cloud_dict['D_mass'],
            'mmd_c':cloud_dict['mmd'],
            'D_num_r':rain_dict['D_num'],
            'D_vol_r':rain_dict['D_vol'],
            'D_mass_r':rain_dict['D_mass'],
            'mmd_r':rain_dict['mmd'],
            #'sigma_num_c':cloud_dict['sigma_num'],
            #'skew_num_c':cloud_dict['skew_num'],
            #'kurt_num_c':cloud_dict['kurt_num'],
            #'sigma_mass_c':cloud_dict['sigma_mass'],
            #'skew_mass_c':cloud_dict['skew_mass'],
            #'kurt_mass_c':cloud_dict['kurt_mass'],
            #'sigma_num_r':rain_dict['sigma_num'],
            #'skew_num_r':rain_dict['skew_num'],
            #'kurt_num_r':rain_dict['kurt_num'],
            #'sigma_mass_r':rain_dict['sigma_mass'],
            #'skew_mass_r':rain_dict['skew_mass'],
            #'kurt_mass_r':rain_dict['kurt_mass'],
            'q_r_gt_100um':rain_dict['q_gt_100um'],
            'N_r_gt_100um':rain_dict['N_gt_100um'],
            'q_c_gt_100um':cloud_dict['q_gt_100um'],
            'N_c_gt_100um':cloud_dict['N_gt_100um'],
               }

    # Define variable attributes
    out_dict_attrs = {
                'D_num': {
                    'long_name': 'Number-weighted Mean Diameter',
                    'units': 'm',
                },
                'D_vol':{
                    'long_name': 'Mean Volumetric Diameter',
                    'units':'m',
                },
                'D_mass':{
                    'long_name': 'Mass-weighted Mean Diameter',
                    'units':'m',
                },
                #'sigma_num':{
                #    'long_name': 'Number-weighted DSD Standard Deviation',
                #    'units':'m',
                #    'description_1':'To derive relative dispersion, divide sigma_num by D_num',
                #},
                #'skew_num':{
                #    'long_name': 'Number-weighted DSD Skewness',
                #    'units':'unitless',
                #},
                #'kurt_num':{
                #    'long_name': 'Number-weighted DSD Kurtosis',
                #    'units':'Dimensionless',
                #},
                #'sigma_mass':{
                #    'long_name': 'Mass-weighted DSD Standard Deviation',
                #    'units':'m',
                #    'description_1':'To derive relative dispersion, divide sigma_num by D_mass',
                #},
                #'skew_mass':{
                #    'long_name': 'Mass-weighted DSD Skewness',
                #    'units':'unitless',
                #},
                #'kurt_mass':{
                #    'long_name': 'Mass-weighted DSD Kurtosis',
                #    'units':'Dimensionless',
                #},
                'mmd':{
                    'long_name': 'DSD Median Mass Diameter',
                    'units':'m',
                },
                'N_gt_100um':{
                    'long_name': 'Total Number Concentration (Cloud + Rain) Mixing Ratio for Diameters > 100 um',
                    'units':'kg^-1',
                },
                'q_gt_100um':{
                    'long_name': 'Total Mass (Cloud + Rain) Mixing Ratio for Diameters > 100 um',
                    'units':'kg kg^-1',
                },
                'N':{
                    'long_name': 'Total Number Concentration (Cloud + Rain) Mixing Ratio',
                    'units':'kg^-1',
                },
                'q':{
                    'long_name': 'Total Mass (Cloud + Rain) Mixing Ratio',
                    'units':'kg kg^-1',
                },
                'rho_air':{
                    'long_name': 'Moist Air Density',
                    'units':'kg m^-3',
                },
                'temp':{
                    'long_name': 'Temperature',
                    'units':'deg C',
                },
                'N_0_r':{
                    'long_name': 'Rain Intercept Parameter',
                    'units':'m^-4',
                },
                'N_0_c':{
                    'long_name': 'Cloud Intercept Parameter',
                    'units':'m^-4',
                },
                'lambda_r':{
                    'long_name': 'Rain Slope Parameter',
                    'units':'m^-1',
                },
                'lambda_c':{
                    'long_name': 'Cloud Slope Parameter',
                    'units':'m^-1',
                },
                'mu_c':{
                    'long_name': 'Cloud Shape Parameter',
                    'units':'unitless',
                },
                'D_num_c':{
                    'long_name': 'Cloud Number-weighted Mean Diameter',
                    'units':'m',
                },
                'D_vol_c':{
                    'long_name': 'Cloud Volumetric Mean Diameter',
                    'units':'m',
                },
                'D_mass_c':{
                    'long_name': 'Cloud Mass-weighted Mean Diameter',
                    'units':'m',
                },
                'mmd_c':{
                    'long_name': 'Cloud Median Mass Diameter',
                    'units':'m',
                },
                'D_num_r':{
                    'long_name': 'Rain Number-weighted Mean Diameter',
                    'units':'m',
                },
                'D_vol_r':{
                    'long_name': 'Rain Volumetric Mean Diameter',
                    'units':'m',
                },
                'D_mass_r':{
                    'long_name': 'Rain Mass-weighted Mean Diameter',
                    'units':'m',
                },
                'mmd_r':{
                    'long_name': 'Rain Median Mass Diameter',
                    'units':'m',
                },
                #'sigma_num_c':{
                #    'long_name': 'Cloud Number-weighted Standard Deviation',
                #    'units':'m',
                #    'description_1':'To derive relative dispersion, divide sigma_num_c by D_num_c',
                #},
                #'skew_num_c':{
                #    'long_name': 'Cloud Number-weighted Skewness',
                #    'units':'unitless',
                #},
                #'kurt_num_c':{
                #    'long_name': 'Cloud Number-weighted Kurtosis',
                #    'units':'unitless',
                #},
                #'sigma_mass_c':{
                #    'long_name': 'Cloud Mass-weighted Standard Deviation',
                #    'units':'m',
                #    'description_1':'To derive relative dispersion, divide sigma_mass_c by D_mass_c',
                #},
                #'skew_mass_c':{
                #    'long_name': 'Cloud Mass-weighted Skewness',
                #    'units':'unitless',
                #},
                #'kurt_mass_c':{
                #    'long_name': 'Cloud Mass-weighted Kurtosis',
                #    'units':'unitless',
                #},
                #'sigma_num_r':{
                #    'long_name': 'Rain Number-weighted Standard Deviation',
                #    'units':'m',
                #    'description_1':'To derive relative dispersion, divide sigma_num_r by D_num_r',
                #},
                #'skew_num_r':{
                #    'long_name': 'Rain Number-weighted Skewness',
                #    'units':'unitless',
                #},
                #'kurt_num_r':{
                #    'long_name': 'Rain Number-weighted Kurtosis',
                #    'units':'unitless',
                #},
                #'sigma_mass_r':{
                #    'long_name': 'Rain Mass-weighted Standard Deviation',
                #    'units':'m',
                #    'description_1':'To derive relative dispersion, divide sigma_mass_r by D_mass_r',
                #},
                #'skew_mass_r':{
                #    'long_name': 'Rain Mass-weighted Skewness',
                #    'units':'unitless',
                #},
                #'kurt_mass_r':{
                #    'long_name': 'Rain Mass-weighted Kurtosis',
                #    'units':'unitless',
                #},
                'q_r_gt_100um':{
                    'long_name': 'Rain Mass Mixing Ratio for Diameters > 100 um',
                    'units':'kg/kg',
                },
                'N_r_gt_100um':{
                    'long_name': 'Rain Number Concentration Mixing Ratio for Diameters > 100 um',
                    'units':'/kg',
                },
                'q_c_gt_100um':{
                    'long_name': 'Cloud Mass Mixing Ratio for Diameters > 100 um',
                    'units':'kg/kg',
                },
                'N_c_gt_100um':{
                    'long_name': 'Cloud Number Concentration Mixing Ratio for Diameters > 100 um',
                    'units':'/kg',
                },

    }   

    date_str = x3d_dict['time_dt'].strftime('%Y%m%d_%H:%M:%S')
    # Output parameters
    output_path = f'/pscratch/sd/m/mckenna/wrf_lasso/coarse_grained/{sim_date}/{ens}/{domain}/derived/'
    output_filename = f'{output_path}LASSO_CACTI_2.5km_DSD_parameters_liq_only_{date_str}_{ens}_{domain}_native.nc'


    
    # Define dimensions
    time_dimname = 'time'
    lon_dimname = 'west_east'
    lat_dimname = 'south_north'
    z_dimname = 'bottom_top'

    var_dict = {}
    # Define output variable dictionary
    for key, value in out_dict.items():
        if np.ndim(value) == 3.:
            var_dict[key] = ([z_dimname,lat_dimname,lon_dimname],value,out_dict_attrs[key])
            
    wrf_epoch = np.asarray([int(x3d_dict['time_dt'].timestamp())])

    # Define coordinate attributes
    coord_attr_dict = {'time':{'long_name':'WRF Epoch time','units':'Epoch'},
                       'lon':{'long_name':'Longitude','units':'degrees'},
                       'lat':{'long_name':'Latitude','units':'degrees'},
                       'zamsl':{'long_name':'Height Above Mean Sea Level','units':'m'},
                      }
    # Define coordinates
    coord_dict = {
            'time': ([time_dimname],wrf_epoch,coord_attr_dict['time']),
            'lon': ([lat_dimname,lon_dimname],x3d_dict['lon'],coord_attr_dict['lon']),
            'lat': ([lat_dimname,lon_dimname],x3d_dict['lat'],coord_attr_dict['lat']),
            'zamsl': ([z_dimname,lat_dimname,lon_dimname],x3d_dict['zamsl'],coord_attr_dict['zamsl']),
            }

    # Define global attributes
    gattr_dict = {
        'title':  f'WRF-LASSO derived 2.5-km liquid DSD parameters for coarse-grained domain {domain}', \
        'description_1':  f'Date: {sim_date}', \
        'description_2':  f'Ensemble Member: {ens}', \
        'Institution': 'Pacific Northwest National Laboratoy', \
        'Contact': 'McKenna Stanford, mckenna.stnaford@pnnl.gov', \
        'Created_on':  time.ctime(time.time()), \
        'Date': date_str, \
    }

    # Define xarray dataset
    dsout = xr.Dataset(var_dict, coords=coord_dict, attrs=gattr_dict)
    
    # Delete file if it already exists
    if os.path.isfile(output_filename):
        os.remove(output_filename)
    
    # Set encoding/compression for all variables
    comp = dict(zlib=True)
    encoding = {var: comp for var in dsout.data_vars}
    
    # Write to netcdf file
    dsout.to_netcdf(path=output_filename, mode="w",
                    format="NETCDF4", unlimited_dims=time_dimname, encoding=encoding)

    return output_filename

In [22]:
def compute_liquid_dsd_parameters(cloud_dict, rain_dict, rho_air, q_min_threshold=1e-12, N_min_threshold=1e-6):
    """
    Computes combined liquid (cloud water + rain) DSD metrics, including DSD moments, number- and -mass-weighted mean diamters, 
    median mass diameter, and reflectivity.

    
    Parameters:
    - cloud_dict: dictionary holding all derived DSD variables for cloud water species
    - rain_dict: dictionary holding all derived DSD variables for rain species
    - rho_air: 3D array of moist air density [kg/m^3]
    
    Returns:
    - Dictionary containing 3D arrays for D_num, D_vol, D_mass, 
      spectral widths, skewnesses, kurtosis, median mass diameter, and reflectivity (Z)
    """

    # Combine rain and cloud species data
    M_comb = {n: np.nansum([rain_dict['M_dict'][n], cloud_dict['M_dict'][n]], axis=0) for n in rain_dict['M_dict']}
    q_comb = np.nansum([rain_dict['q'],cloud_dict['q']],axis=0)
    N_comb = np.nansum([rain_dict['N'],cloud_dict['N']],axis=0)
    Z_comb = np.nansum([rain_dict['Z'],cloud_dict['Z']],axis=0)
    #N_r = rain_dict['N']*rho_air
    #N_c = cloud_dict['N']*rho_air
    #q_r = rain_dict['q']*rho_air
    #q_c = cloud_dict['q']*rho_air
    #print('np.nanmax(N_comb) [m^-3]:',np.nanmax(N_comb*rho_air))
    #print('np.nanmax(N_r) [m^-3]:',np.nanmax(N_r))
    #print('np.nanmax(N_c) [m^-3]:',np.nanmax(N_c))
    #print('np.nanmax(q_comb) [g m^-3]:',np.nanmax(q_comb*rho_air*1.e3))
    #print('np.nanmax(q_r) [g m^-3]:',np.nanmax(q_r)*1.e3)
    #print('np.nanmax(q_c) [g m^-3]:',np.nanmax(q_c)*1.e3)
    #print('')
    
    # Extract parameters
    lambda_c, N_0_c, mu_c = cloud_dict['lambda'], cloud_dict['N_0'], cloud_dict['mu']
    lambda_r, N_0_r, mu_r = rain_dict['lambda'], rain_dict['N_0'], rain_dict['mu']

    # Constants
    rho_w = 1000. # kg/m^3
    a_r = a_c = np.pi*rho_w/6.
    b_r = b_c = 3.

    rho_air_orig = rho_air.copy()
    
    #valid_mask = ~np.isnan(M_comb[0])  # True where M0 is valid (not NaN)
    #valid_mask = np.where(q_comb > 1.e-4)
    #valid_mask = np.where(q_comb*rho_air > 1.e-12)
    valid_mask = np.where(q_comb > 1.e-12)

    # Apply mask    
    N_0_r, N_0_c = N_0_r[valid_mask], N_0_c[valid_mask]
    lambda_r, lambda_c = lambda_r[valid_mask], lambda_c[valid_mask]
    mu_c = mu_c[valid_mask]
    rho_air = rho_air[valid_mask]
    Z_ij = Z_comb[valid_mask]
    #N_r = N_r[valid_mask]
    #q_r = q_r[valid_mask]
    #N_c = N_c[valid_mask]
    #q_c = q_c[valid_mask]
    #print('np.nanmax(N_r) [m^-3]:',np.nanmax(N_r))
    #print('np.nanmax(N_c) [m^-3]:',np.nanmax(N_c))
    #print('np.nanmax(q_r) [g/m^-3]:',np.nanmax(q_r)*1.e3)
    #print('np.nanmax(q_c) [g/m^3]:',np.nanmax(q_c)*1.e3)
    #print('

    # Initialize arrays with NaN values
    D_num = np.full_like(M_comb[0], np.nan, dtype=np.float64)
    D_vol = np.full_like(M_comb[0], np.nan, dtype=np.float64)
    D_mass = np.full_like(M_comb[0], np.nan, dtype=np.float64)
    #sigma_num = np.full_like(M_comb[0], np.nan, dtype=np.float64)
    #skew_num = np.full_like(M_comb[0], np.nan, dtype=np.float64)
    #kurt_num = np.full_like(M_comb[0], np.nan, dtype=np.float64)
    #sigma_mass = np.full_like(M_comb[0], np.nan, dtype=np.float64)
    #skew_mass = np.full_like(M_comb[0], np.nan, dtype=np.float64)
    #kurt_mass = np.full_like(M_comb[0], np.nan, dtype=np.float64)
    median_mass_diameter = np.full_like(M_comb[0], np.nan, dtype=np.float64)
    Z = np.full_like(M_comb[0], np.nan, dtype=np.float64)
    dBZ = np.full_like(M_comb[0], np.nan, dtype=np.float64)
    N_gt_100um = np.full_like(M_comb[0], np.nan, dtype=np.float64)
    q_gt_100um = np.full_like(M_comb[0], np.nan, dtype=np.float64)

    # Extract valid combined moments
    M_comb_ij = {}
    for key in M_comb:
        M_comb_ij[key] = M_comb[key][valid_mask]

    # Number-weighted statistics (D_num, D_vol, sigma_num, skew_num, kurt_num)
    D_num_ij = M_comb_ij[1] / M_comb_ij[0]
    D_vol_ij = (M_comb_ij[3] / M_comb_ij[0]) ** (1/3)
    
    # Central moments (number-weighted)
    mu2_num = M_comb_ij[2] / M_comb_ij[0] - D_num_ij**2
    mu3_num = M_comb_ij[3] / M_comb_ij[0] - 3 * D_num_ij * mu2_num - D_num_ij**3
    mu4_num = M_comb_ij[4] / M_comb_ij[0] - 4 * D_num_ij * mu3_num - 6 * D_num_ij**2 * mu2_num - D_num_ij**4
    
    #sigma_num_ij = np.sqrt(mu2_num)
    #skew_num_ij = mu3_num / sigma_num_ij**3
    #kurt_num_ij = mu4_num / sigma_num_ij**4 - 3.0  # excess kurtosis
    
    # Mass-weighted statistics (D_mass, sigma_mass, skew_mass, kurt_mass)
    D_mass_ij = M_comb_ij[4] / M_comb_ij[3]
    
    # Central moments (mass-weighted)
    mu2_mass = M_comb_ij[5] / M_comb_ij[3] - D_mass_ij**2
    mu3_mass = M_comb_ij[6] / M_comb_ij[3] - 3 * D_mass_ij * mu2_mass - D_mass_ij**3
    mu4_mass = M_comb_ij[7] / M_comb_ij[3] - 4 * D_mass_ij * mu3_mass - 6 * D_mass_ij**2 * mu2_mass - D_mass_ij**4
    
    #sigma_mass_ij = np.sqrt(mu2_mass)
    #skew_mass_ij = mu3_mass / sigma_mass_ij**3
    #kurt_mass_ij = mu4_mass / sigma_mass_ij**4 - 3.0  # excess kurtosis
    
    # Median mass diameter
    D_bins = np.arange(-0.5,10001.5,1)*1.e-6 # [m]
    D = np.convolve(D_bins, np.ones(2) / 2, 'valid')  # Midpoint of D bins
    dD = np.diff(D_bins)[0]  # constant bin width

    # Broadcast to 2D arrays: (num_grid_points, num_D)
    D_broadcast = D[None, :]  # Shape: (1, num_D)
    
    # Set chunk size (adjust as needed to avoid memory overload)
    chunk_size = 10000
    num_grid_points = N_0_r.shape[0]
    print('# of grid points:',num_grid_points)
    
    # Allocate arrays for chunk calculations
    median_mass_diameter_ij, qtot = np.full(num_grid_points, np.nan), np.full(num_grid_points, np.nan)   
    N_gt_100um_ij,q_gt_100um_ij = np.full(num_grid_points, np.nan), np.full(num_grid_points, np.nan)


    # All of the following calculated outside of loop for efficiency
    D_power_mu_b_r = D_broadcast**(b_r + mu_r)
    D_power_mu_r = D_broadcast**(mu_r)
    D_power_b_c = D_broadcast**b_c
    N_0_r = np.nan_to_num(N_0_r, nan=0.0)
    N_0_c = np.nan_to_num(N_0_c, nan=0.0)
    lambda_r = np.nan_to_num(lambda_r, nan=1e12)
    lambda_c = np.nan_to_num(lambda_c, nan=1e12)
    mu_c = np.nan_to_num(mu_c, nan=0.0)

    # Threshold Size
    threshold_size = 100.*1.e-6 # 100 um, in meters
    threshold_mask = D_broadcast > threshold_size   


    for i_start in range(0, num_grid_points, chunk_size):
        #print('------------------------')
        i_end = min(i_start + chunk_size, num_grid_points)

        # Progress print
        print(f"Progress: {100 * i_end / num_grid_points:.1f}%", end='\r')
    
        # Slice chunks
        N_0_r_chunk = N_0_r[i_start:i_end, None]
        N_0_c_chunk = N_0_c[i_start:i_end, None]
        lambda_r_chunk = lambda_r[i_start:i_end, None]
        lambda_c_chunk = lambda_c[i_start:i_end, None]
        mu_c_chunk = mu_c[i_start:i_end, None]

        D_power_mu_b_c = D_power_b_c * D_broadcast**mu_c_chunk
        D_power_mu_c = D_broadcast**mu_c_chunk
        # Compute M_r and M_c for this chunk
        M_r_chunk = a_r * N_0_r_chunk * D_power_mu_b_r * np.exp(-lambda_r_chunk * D_broadcast)
        M_c_chunk = a_c * N_0_c_chunk * D_power_mu_b_c * np.exp(-lambda_c_chunk * D_broadcast)
        # Compute N_r and N_c for this chunk
        N_r_chunk = N_0_r_chunk * D_power_mu_r * np.exp(-lambda_r_chunk * D_broadcast)
        N_c_chunk = N_0_c_chunk * D_power_mu_c * np.exp(-lambda_c_chunk * D_broadcast)
        #N_r_chunk_2 = N_r[i_start:i_end]
        #N_c_chunk_2 = N_c[i_start:i_end]
        #M_r_chunk_2 = q_r[i_start:i_end]
        #M_c_chunk_2 = q_c[i_start:i_end]

        #int_N_r_chunk = np.trapz(N_r_chunk,D,axis=1)
        #int_N_c_chunk = np.trapz(N_c_chunk,D,axis=1)
        #int_M_r_chunk = np.trapz(M_r_chunk,D,axis=1)
        #int_M_c_chunk = np.trapz(M_c_chunk,D,axis=1)
 
        #print('np.nanmax(N_r_chunk_2) [m^-3]:',np.nanmax(N_r_chunk_2))
        #print('np.nanmax(int_N_r_chunk) [m^-3]:',np.nanmax(int_N_r_chunk))
        #print('np.nanmax(N_c_chunk_2) [m^-3]:',np.nanmax(N_c_chunk_2))
        #print('np.nanmax(int_N_c_chunk) [m^-3]:',np.nanmax(int_N_c_chunk))
        #print('')
        #print('np.nanmax(M_r_chunk_2) [g m^-3]:',np.nanmax(M_r_chunk_2)*1.e3)
        #print('np.nanmax(int_M_r_chunk) [g m^-3]:',np.nanmax(int_M_r_chunk)*1.e3)
        #print('np.nanmax(M_c_chunk_2) [g m^-3]:',np.nanmax(M_c_chunk_2)*1.e3)
        #print('np.nanmax(int_M_c_chunk) [g m^-3]:',np.nanmax(int_M_c_chunk)*1.e3)
        #print('')
        
        M_total_chunk = M_r_chunk + M_c_chunk
    
        # Example: Compute cumulative mass distribution and median mass diameter
        M_cumsum = np.cumsum(M_total_chunk * dD, axis=1)
        M_total_integrated = M_cumsum[:, -1]
        qtot[i_start:i_end] = M_total_integrated
        mask = M_total_integrated > 0
        M_fraction = M_cumsum[mask] / M_total_integrated[mask, None]

        # Find median diameter index
        D_median_chunk = np.full(i_end - i_start, np.nan)
        idx_median = np.argmax(M_fraction >= 0.5,axis=1)
        D_median_chunk[mask] = D[idx_median]
        median_mass_diameter_ij[i_start:i_end] = D_median_chunk



        # Calculate number concentration for particles > 100 um
        N_r_gt_100um_chunk = np.sum(N_r_chunk * threshold_mask * dD, axis=1)
        N_c_gt_100um_chunk = np.sum(N_c_chunk * threshold_mask * dD, axis=1)
        N_gt_100um_ij[i_start:i_end] = N_r_gt_100um_chunk + N_c_gt_100um_chunk
        #print('np.nanmax(N_gt_100um_ij):',np.nanmax(N_gt_100um_ij))

        # Calculate mass for particles > 100 µm
        q_r_gt_100um_chunk = np.sum(M_r_chunk * threshold_mask * dD, axis=1)
        q_c_gt_100um_chunk = np.sum(M_c_chunk * threshold_mask * dD, axis=1)
        q_gt_100um_ij[i_start:i_end] = q_r_gt_100um_chunk + q_c_gt_100um_chunk


    
    # Store values in the valid array positions
    D_num[valid_mask] = D_num_ij
    D_vol[valid_mask] = D_vol_ij
    D_mass[valid_mask] = D_mass_ij
    #sigma_num[valid_mask] = sigma_num_ij
    #skew_num[valid_mask] = skew_num_ij
    #kurt_num[valid_mask] = kurt_num_ij
    #sigma_mass[valid_mask] = sigma_mass_ij
    #skew_mass[valid_mask] = skew_mass_ij
    #kurt_mass[valid_mask] = kurt_mass_ij
    median_mass_diameter[valid_mask] = median_mass_diameter_ij
    N_gt_100um[valid_mask] = N_gt_100um_ij/rho_air
    #print('np.nanmax(N_gt_100um_ij/rho_air):',np.nanmax(N_gt_100um_ij/rho_air))
    q_gt_100um[valid_mask] = q_gt_100um_ij/rho_air
    Z[valid_mask] = Z_ij
    dBZ[valid_mask] = 10.*np.log10(Z_ij)

    
    return {
        'D_num': D_num,  # Number-weighted mean diameter [m]
        'D_vol': D_vol,  # Volume-weighted mean diameter [m]
        'D_mass': D_mass,  # Mass-weighted mean diameter [m]
        #'sigma_num': sigma_num,  # Number-weighted standard deviation [m]
        #'skew_num': skew_num,  # Number-weighted skewness [-]
        #'kurt_num': kurt_num,  # Number-weighted kurtosis [-]
        #'sigma_mass': sigma_mass,  # Mass-weighted standard deviation [-]
        #'skew_mass': skew_mass,  # Mass-weighted skewness [-]
        #'kurt_mass': kurt_mass,  # Mass-weighted kurtosis [-]
        'mmd': median_mass_diameter,  # Median mass diameter [m]
        'N_gt_100um': N_gt_100um,  # Number concentration mixing ratio for diameters > 100 um [kg^-1]
        'q_gt_100um': q_gt_100um,  # Mass mixing ratio for diameters > 100 um [kg kg^-1]
        'Z': Z,  # Linear Reflectivity [mm^6 m^-3]
        'dBZ': dBZ,  # Reflectivity [dBZ]
        'q':q_comb, # Sum of cloud and rain mass mixing ratios [kg/kg]
        'N':N_comb, # Sum of cloud and rain number concentraiton mixing ratio [#/kg]
        'rho_air':rho_air_orig, # moist air density [kg/m^3]
        'M_dict':M_comb, # dictionary of raw moments
    }

# Test in serial

In [15]:
%%time
if True:
    dumi = 10
    #x3d_dict = get_3d_vars(wrf_3d_files[dumi],z_agl,zamsl,csapr_mask,lon_2d,lat_2d)
    x3d_dict = get_3d_vars(cld_files[dumi],met_files[dumi],csapr_files[0])
    rain_dict = compute_gamma_dsd_parameters_liquid_3d(x3d_dict['qr'], x3d_dict['nr'], x3d_dict['rho_air'], species='rain', mu=0.0, rho_w=1000.0, q_min_threshold=1e-12, N_min_threshold=1e-6)
    cloud_dict = compute_gamma_dsd_parameters_liquid_3d(x3d_dict['qc'], x3d_dict['nc'], x3d_dict['rho_air'], species='cloud', mu=0.0, rho_w=1000.0, q_min_threshold=1e-12, N_min_threshold=1e-6)
    liq_dict = compute_liquid_dsd_parameters(cloud_dict, rain_dict, x3d_dict['rho_air'], q_min_threshold=1e-12, N_min_threshold=1e-6)
    dsout,outfile_name = write_file(x3d_dict,rain_dict,cloud_dict,liq_dict)
    #outfile_name = driver_func(wrf_3d_files[dumi])

# of grid points: 6269
CPU times: user 11.4 s, sys: 1.67 s, total: 13 s
Wall time: 15.3 s


In [16]:
dsout

<xarray.Dataset> Size: 235MB
Dimensions:       (bottom_top: 149, south_north: 87, west_east: 89, time: 1)
Coordinates:
  * time          (time) int64 8B 1543501800
    lon           (south_north, west_east) float32 31kB -65.92 -65.9 ... -63.57
    lat           (south_north, west_east) float32 31kB -33.1 -33.1 ... -31.15
    zamsl         (bottom_top, south_north, west_east) float32 5MB 1.031e+03 ...
Dimensions without coordinates: bottom_top, south_north, west_east
Data variables: (12/27)
    D_num         (bottom_top, south_north, west_east) float64 9MB nan ... nan
    D_vol         (bottom_top, south_north, west_east) float64 9MB nan ... nan
    D_mass        (bottom_top, south_north, west_east) float64 9MB nan ... nan
    mmd           (bottom_top, south_north, west_east) float64 9MB nan ... nan
    N_gt_100um    (bottom_top, south_north, west_east) float64 9MB nan ... nan
    q_gt_100um    (bottom_top, south_north, west_east) float64 9MB nan ... nan
    ...            ...
    D_mass_r      (bottom_top, south_north, west_east) float64 9MB nan ... nan
    mmd_r         (bottom_top, south_north, west_east) float64 9MB nan ... nan
    q_r_gt_100um  (bottom_top, south_north, west_east) float64 9MB nan ... nan
    N_r_gt_100um  (bottom_top, south_north, west_east) float64 9MB nan ... nan
    q_c_gt_100um  (bottom_top, south_north, west_east) float64 9MB nan ... nan
    N_c_gt_100um  (bottom_top, south_north, west_east) float64 9MB nan ... nan
Attributes:
    title:          WRF-LASSO derived 2.5-km liquid DSD parameters for coarse...
    description_1:  Date: 20181129
    description_2:  Ensemble Member: gefs03
    Institution:    Pacific Northwest National Laboratoy
    Contact:        McKenna Stanford, mckenna.stnaford@pnnl.gov
    Created_on:     Wed Dec 31 11:15:51 2025
    Date:           20181129_14:30:00

# Dask stuff

In [17]:
dask.config.config["distributed"]["dashboard"]["link"] = "{JUPYTERHUB_SERVICE_PREFIX}proxy/{host}:{port}/status" 

In [18]:
iparallel = True
if iparallel:
    cluster = LocalCluster(n_workers=20,threads_per_worker=1,memory_limit='20GB')#,dashboard_address=':8787')
    client = Client(cluster)

In [19]:
cluster

LocalCluster(4cd1233a, 'tcp://127.0.0.1:36851', workers=20, threads=20, memory=372.53 GiB)

In [20]:
def suppress_warnings():
    import warnings
    warnings.filterwarnings("ignore", message=".*pyproj unable to set PROJ database path.*", category=UserWarning)
    warnings.filterwarnings("ignore", category=RuntimeWarning)

client.run(suppress_warnings)

{'tcp://127.0.0.1:34351': None,
 'tcp://127.0.0.1:34701': None,
 'tcp://127.0.0.1:34797': None,
 'tcp://127.0.0.1:34877': None,
 'tcp://127.0.0.1:35199': None,
 'tcp://127.0.0.1:35341': None,
 'tcp://127.0.0.1:35743': None,
 'tcp://127.0.0.1:38421': None,
 'tcp://127.0.0.1:38957': None,
 'tcp://127.0.0.1:41143': None,
 'tcp://127.0.0.1:42471': None,
 'tcp://127.0.0.1:44081': None,
 'tcp://127.0.0.1:44721': None,
 'tcp://127.0.0.1:44797': None,
 'tcp://127.0.0.1:44929': None,
 'tcp://127.0.0.1:45035': None,
 'tcp://127.0.0.1:45221': None,
 'tcp://127.0.0.1:46251': None,
 'tcp://127.0.0.1:46395': None,
 'tcp://127.0.0.1:46877': None}

In [15]:
#cluster.scale(n=14)

In [23]:
def driver_func(cld_file,met_file,csapr_file):
    x3d_dict = get_3d_vars(cld_file,met_file,csapr_file)
    rain_dict = compute_gamma_dsd_parameters_liquid_3d(x3d_dict['qr'], x3d_dict['nr'], x3d_dict['rho_air'], species='rain', mu=0.0, rho_w=1000.0, q_min_threshold=1e-12, N_min_threshold=1e-6)
    cloud_dict = compute_gamma_dsd_parameters_liquid_3d(x3d_dict['qc'], x3d_dict['nc'], x3d_dict['rho_air'], species='cloud', mu=0.0, rho_w=1000.0, q_min_threshold=1e-12, N_min_threshold=1e-6)
    liq_dict = compute_liquid_dsd_parameters(cloud_dict, rain_dict, cloud_dict['rho_air'], q_min_threshold=1e-12, N_min_threshold=1e-6)
    outfile_name = write_file(x3d_dict,rain_dict,cloud_dict,liq_dict)
    return outfile_name

In [59]:
outfile_name = driver_func(cld_files[0],met_files[0],csapr_files[0])

# of grid points: 201
Progress: 100.0%

In [24]:
sim_dates = ['20181129','20181204','20181205','20181219','20190122','20190125','20190129','20190208']
enss = ['gefs03','gefs18','gefs01','eda09','gefs01','eda07','eda09','eda03']
domains = ['d2','d3','d4']

#for ix in range(len(sim_dates)):
for ix in range(len(sim_dates)):
    sim_date = sim_dates[ix]
    ens = enss[ix]
    print('sim_date:',sim_date)
    for jx in range(len(domains)):
        domain = domains[jx]
        print('domain:',domain)


        path = f'/pscratch/sd/m/mckenna/wrf_lasso/coarse_grained/{sim_date}/{ens}/{domain}/cld_met_files/'
        
        cld_files = sorted(glob.glob(path+'*_cld_*.nc'))#[::3]
        met_files = sorted(glob.glob(path+'*_met_*.nc'))#[::3]
        num_cld_files = len(cld_files)
        num_met_files = len(met_files)
        print('# of cld files:',num_cld_files)
        print('# of met files:',num_met_files)
        
        cld_datetimes = []
        for ii in range(num_cld_files):
            #print(cld_files[ii])
            if domain != 'd2':
                dum_str = cld_files[ii].split('/')[-1].split('_')[-1].split('.')
                date_str = dum_str[2]
                time_str = dum_str[3]
                cld_datetimes.append(datetime.datetime(int(date_str[0:4]),int(date_str[4:6]),int(date_str[6:8]),int(time_str[0:2]),int(time_str[2:4]),0,tzinfo=datetime.timezone.utc))
            else:
                dum_str = cld_files[ii].split('/')[-1].split('.')
                date_str = dum_str[-3]
                time_str = dum_str[-2]
                cld_datetimes.append(datetime.datetime(int(date_str[0:4]),int(date_str[4:6]),int(date_str[6:8]),int(time_str[0:2]),int(time_str[2:4]),0,tzinfo=datetime.timezone.utc))

        results = []
        for ii in range(num_cld_files):
            result = dask.delayed(driver_func)(cld_files[ii],met_files[ii],csapr_files[0])
            results.append(result)
        # Trigger dask computation
        final_result = dask.compute(*results)
        wait(final_result)


sim_date: 20181129
domain: d2
# of cld files: 49
# of met files: 49
domain: d3
# of cld files: 48
# of met files: 48
domain: d4
# of cld files: 49
# of met files: 49
sim_date: 20181204
domain: d2
# of cld files: 49
# of met files: 49
domain: d3
# of cld files: 48
# of met files: 48
domain: d4
# of cld files: 49
# of met files: 49
sim_date: 20181205
domain: d2
# of cld files: 49
# of met files: 49
domain: d3
# of cld files: 48
# of met files: 48
domain: d4
# of cld files: 49
# of met files: 49
sim_date: 20181219
domain: d2
# of cld files: 49
# of met files: 49
domain: d3
# of cld files: 48
# of met files: 48
domain: d4
# of cld files: 49
# of met files: 49
sim_date: 20190122
domain: d2
# of cld files: 49
# of met files: 49
domain: d3
# of cld files: 48
# of met files: 48
domain: d4
# of cld files: 49
# of met files: 49
sim_date: 20190125
domain: d2
# of cld files: 49
# of met files: 49
domain: d3
# of cld files: 48
# of met files: 48
domain: d4
# of cld files: 49
# of met files: 49
sim_

In [60]:
print(outfile_name)

/pscratch/sd/m/mckenna/wrf_lasso/coarse_grained/20181129/gefs03/d2/derived/LASSO_CACTI_2.5km_DSD_parameters_liq_only_20181129_12:00:00_gefs03_d2_native.nc


In [33]:
%%time
results = []
start_id = 0
#end_id = 100
end_id = len(wrf_3d_files)
for ii in range(start_id,end_id):
    result = dask.delayed(driver_func)(wrf_3d_files[ii])
    results.append(result)

CPU times: user 341 ms, sys: 87.9 ms, total: 429 ms
Wall time: 366 ms


In [34]:
%%time
futures = client.compute(results)  # results is a list of delayed or dask collections

CPU times: user 281 ms, sys: 18.8 ms, total: 299 ms
Wall time: 297 ms


In [36]:
%%time
# Trigger dask computation
#with warnings.catch_warnings():
#    warnings.simplefilter("ignore", category=RuntimeWarning)
#final_result = dask.compute(*results)
final_result = client.gather(futures)
print("Computation done.")
#wait(final_result)

Computation done.
CPU times: user 633 ms, sys: 169 ms, total: 802 ms
Wall time: 820 ms


In [23]:
len(final_result)

5000

In [24]:
final_result[0]

'/pscratch/sd/m/mckenna/cacti/wrf/derived_3km_dsd_parameters/WRF_CACTI_3km_DSD_parameters_liq_only_20181015_00:00:00.nc'

In [37]:
# Gracefully close previous client and cluster if they exist
try:
    client.close()
except NameError:
    pass

try:
    cluster.close()
except NameError:
    pass

# of grid points: 36534
# of grid points: 1030
# of grid points: 1869
# of grid points: 151931
# of grid points: 15728
# of grid points: 56486
# of grid points: 92557
 0ogress: 100.0%
# of grid points: 966
# of grid points: 26483
# of grid points: 92467
 4916ess: 100.0%
# of grid points: 0
# of grid points: 1076
# of grid points: 0
# of grid points: 106669
# of grid points: 104070
# of grid points: 13074
# of grid points: 3864
# of grid points: 154194
# of grid points: 55000
# of grid points: 0
# of grid points: 6721
# of grid points: 47114
# of grid points: 37940
# of grid points: 10
# of grid points: 30698
# of grid points: 71079
# of grid points: 1350
# of grid points: 58
# of grid points: 5038
# of grid points: 100974
# of grid points: 82849
# of grid points: 0
# of grid points: 0
# of grid points: 17
# of grid points: 2925
# of grid points: 49
# of grid points: 71847
# of grid points: 0
# of grid points: 43459
# of grid points: 0
# of grid points: 0
# of grid points: 97607
# of gr